In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
import os

In [ ]:
method_train_summary = {}
method_test_summary = {}

for feature_num in range(1,51):

    species_id = 'ZWS'
    dir_path = f'../phd_thesis/103_leave/{species_id}'
    feature_df = pd.read_csv(os.path.join(dir_path, 'df_feature.csv'), index_col = 0)
    meta_df = pd.read_csv(os.path.join(dir_path, 'df_meta.csv'),index_col = 0)
    summary_df = pd.read_csv(os.path.join(dir_path, 'df_summary.csv'),index_col = 0)
    assert (meta_df.index == feature_df.index).all()
    assert (summary_df.index == feature_df.columns).all()

    train_species = meta_df.index.to_list()
    train_species.remove(species_id)

    
    # 修正原代码拼写错误(train_speices)及切片语法，取前30列特征
    train_feature = feature_df.loc[train_species, :].iloc[:, :feature_num]
    train_label   = meta_df.loc[train_species, 'label'].values

    # 全量特征（含 species_id），用于最终全量预测

    print(f'{feature_num} ==============================================')
    all_feature = feature_df.iloc[:, :feature_num]

    from sklearn.linear_model import LogisticRegression
    from sklearn.naive_bayes import CategoricalNB   # 适合分类型特征（氨基酸类型）的贝叶斯模型
    from sklearn.svm import SVC
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.preprocessing import OrdinalEncoder
    from sklearn.metrics import accuracy_score

    # ---- Encoding：每列为氨基酸字符串，用 OrdinalEncoder 转换为整数 ----
    # 在全量数据上 fit，确保测试集类别均可编码
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    encoder.fit(feature_df.iloc[:, :feature_num])

    X_train = encoder.transform(train_feature)
    X_all   = encoder.transform(all_feature)
    y_train = train_label
    y_all   = meta_df['label'].values

    # CategoricalNB 要求非负整数，将 -1（未知）替换为 0
    # CategoricalNB 要求非负整数；X_train_nb 直接转换（训练集无未知类别）
    X_train_nb = X_train.astype(int)
    # X_all 中可能出现训练集未见过的氨基酸（编码为 -1），
    # 需 clip 到每列训练集最大值，防止 CategoricalNB 预测时索引越界
    nb_max_per_col = X_train_nb.max(axis=0)  # shape: (n_features,)
    X_all_nb = np.clip(X_all, 0, nb_max_per_col).astype(int)

    # ---- 四种分类模型 ----
    models = {
        'LogisticRegression':      LogisticRegression(max_iter=1000, random_state=42),
        'NaiveBayes(Categorical)': CategoricalNB(),
        'SVM':                     SVC(kernel='rbf', random_state=42),
        'RandomForest':            RandomForestClassifier(n_estimators=100, random_state=42),
    }

    results = {}

    for model_name, model in models.items():
        # CategoricalNB 使用 clip 后的整数特征，其余使用 OrdinalEncoded 浮点特征
        if model_name == 'NaiveBayes(Categorical)':
            Xtr, Xall = X_train_nb, X_all_nb
        else:
            Xtr, Xall = X_train, X_all

        model.fit(Xtr, y_train)

        # 训练集预测 ACC
        train_pred = model.predict(Xtr)
        train_acc  = accuracy_score(y_train, train_pred)

        # 全量预测（含 species_id 物种）
        all_pred    = model.predict(Xall)
        target_pred = all_pred[meta_df.index == species_id][0]
        target_true = meta_df.loc[species_id, 'label']

        results[model_name] = {
            'train_acc':           train_acc,
            f'{species_id}_pred':  target_pred,
            f'{species_id}_true':  target_true,
            'all_pred':            pd.Series(all_pred, index=feature_df.index),
        }
        # 写入训练集 ACC 字典
        if feature_num not in method_train_summary:
            method_train_summary[feature_num] = {}
        method_train_summary[feature_num][model_name] = train_acc

        # 写入测试集预测结果字典
        if feature_num not in method_test_summary:
            method_test_summary[feature_num] = {}
        method_test_summary[feature_num][model_name] = int(target_pred)

        print(f'[{model_name}]  Train ACC: {train_acc:.4f} | '
            f'{species_id} pred: {target_pred}  (true: {target_true})')

# ---- 循环结束后转为 DataFrame ----
# 行索引为 feature_num，列为各模型名称
train_acc_df = pd.DataFrame(method_train_summary).T.sort_index()
train_acc_df.index.name = 'feature_num'
print("\n训练集 ACC:")
print(train_acc_df)

test_pred_df = pd.DataFrame(method_test_summary).T.sort_index()
test_pred_df.index.name = 'feature_num'
print("\n测试集预测结果:")
print(test_pred_df)


1 ==============================================
[LogisticRegression]  Train ACC: 0.9417 | ZWS pred: 0.0  (true: 1.0)
[NaiveBayes(Categorical)]  Train ACC: 0.9612 | ZWS pred: 0.0  (true: 1.0)
[SVM]  Train ACC: 0.9612 | ZWS pred: 0.0  (true: 1.0)
[RandomForest]  Train ACC: 0.9612 | ZWS pred: 0.0  (true: 1.0)
[RandomForest]  Train ACC: 0.9612 | ZWS pred: 0.0  (true: 1.0)
2 ==============================================
[LogisticRegression]  Train ACC: 0.9320 | ZWS pred: 0.0  (true: 1.0)
[NaiveBayes(Categorical)]  Train ACC: 0.9903 | ZWS pred: 0.0  (true: 1.0)
[SVM]  Train ACC: 0.9903 | ZWS pred: 0.0  (true: 1.0)
[RandomForest]  Train ACC: 0.9903 | ZWS pred: 0.0  (true: 1.0)
2 ==============================================
[LogisticRegression]  Train ACC: 0.9320 | ZWS pred: 0.0  (true: 1.0)
[NaiveBayes(Categorical)]  Train ACC: 0.9903 | ZWS pred: 0.0  (true: 1.0)
[SVM]  Train ACC: 0.9903 | ZWS pred: 0.0  (true: 1.0)
[RandomForest]  Train ACC: 0.9903 | ZWS pred: 0.0  (true: 1.0)
3 ========

In [ ]:
for key in test_pred_df.columns:
    plt.plot(test_pred_df[key].values, label = key)
plt.legend()
plt.show()

for key in test_pred_df.columns:
    plt.plot(train_acc_df[key].values, label = key)
plt.legend()
plt.show()


## 处理各个数据集结果

In [ ]:
species_list = meta_df.index.values
species_label = meta_df.label.values 
species_chinese = meta_df['species_chinese'].values

In [ ]:

for i, species_id in enumerate(species_list):
    print(f'{i} {species_id} {species_chinese[i]} {species_label[i]}=======================================')

    method_train_summary = {}
    method_test_summary = {}

    for feature_num in range(1,31):

        # species_id = 'ZWS'
        dir_path = f'103_leave/{species_id}'
        feature_df = pd.read_csv(os.path.join(dir_path, 'df_feature.csv'), index_col = 0)
        meta_df = pd.read_csv(os.path.join(dir_path, 'df_meta.csv'),index_col = 0)
        summary_df = pd.read_csv(os.path.join(dir_path, 'df_summary.csv'),index_col = 0)
        assert (meta_df.index == feature_df.index).all()
        assert (summary_df.index == feature_df.columns).all()

        train_species = meta_df.index.to_list()
        train_species.remove(species_id)

        
        # 修正原代码拼写错误(train_speices)及切片语法，取前30列特征
        train_feature = feature_df.loc[train_species, :].iloc[:, :feature_num]
        train_label   = meta_df.loc[train_species, 'label'].values

        # 全量特征（含 species_id），用于最终全量预测

        print(f'{feature_num} ==============================================')
        all_feature = feature_df.iloc[:, :feature_num]

        from sklearn.linear_model import LogisticRegression
        from sklearn.naive_bayes import CategoricalNB   # 适合分类型特征（氨基酸类型）的贝叶斯模型
        from sklearn.svm import SVC
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.preprocessing import OrdinalEncoder
        from sklearn.metrics import accuracy_score

        # ---- Encoding：每列为氨基酸字符串，用 OrdinalEncoder 转换为整数 ----
        # 在全量数据上 fit，确保测试集类别均可编码
        encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        encoder.fit(feature_df.iloc[:, :feature_num])

        X_train = encoder.transform(train_feature)
        X_all   = encoder.transform(all_feature)
        y_train = train_label
        y_all   = meta_df['label'].values

        # CategoricalNB 要求非负整数，将 -1（未知）替换为 0
        # CategoricalNB 要求非负整数；X_train_nb 直接转换（训练集无未知类别）
        X_train_nb = X_train.astype(int)
        # X_all 中可能出现训练集未见过的氨基酸（编码为 -1），
        # 需 clip 到每列训练集最大值，防止 CategoricalNB 预测时索引越界
        nb_max_per_col = X_train_nb.max(axis=0)  # shape: (n_features,)
        X_all_nb = np.clip(X_all, 0, nb_max_per_col).astype(int)

        # ---- 四种分类模型 ----
        models = {
            'LogisticRegression':      LogisticRegression(max_iter=1000, random_state=42),
            'NaiveBayes(Categorical)': CategoricalNB(),
            'SVM':                     SVC(kernel='rbf', random_state=42),
            'RandomForest':            RandomForestClassifier(n_estimators=100, random_state=42),
        }

        results = {}

        for model_name, model in models.items():
            # CategoricalNB 使用 clip 后的整数特征，其余使用 OrdinalEncoded 浮点特征
            if model_name == 'NaiveBayes(Categorical)':
                Xtr, Xall = X_train_nb, X_all_nb
            else:
                Xtr, Xall = X_train, X_all

            model.fit(Xtr, y_train)

            # 训练集预测 ACC
            train_pred = model.predict(Xtr)
            train_acc  = accuracy_score(y_train, train_pred)

            # 全量预测（含 species_id 物种）
            all_pred    = model.predict(Xall)
            target_pred = all_pred[meta_df.index == species_id][0]
            target_true = meta_df.loc[species_id, 'label']

            results[model_name] = {
                'train_acc':           train_acc,
                f'{species_id}_pred':  target_pred,
                f'{species_id}_true':  target_true,
                'all_pred':            pd.Series(all_pred, index=feature_df.index),
            }
            # 写入训练集 ACC 字典
            if feature_num not in method_train_summary:
                method_train_summary[feature_num] = {}
            method_train_summary[feature_num][model_name] = train_acc

            # 写入测试集预测结果字典
            if feature_num not in method_test_summary:
                method_test_summary[feature_num] = {}
            method_test_summary[feature_num][model_name] = int(target_pred)

            print(f'[{model_name}]  Train ACC: {train_acc:.4f} | '
                f'{species_id} pred: {target_pred}  (true: {target_true})')

    # ---- 循环结束后转为 DataFrame ----
    # 行索引为 feature_num，列为各模型名称
    train_acc_df = pd.DataFrame(method_train_summary).T.sort_index()
    train_acc_df.index.name = 'feature_num'
    print("\n训练集 ACC:")
    print(train_acc_df)

    test_pred_df = pd.DataFrame(method_test_summary).T.sort_index()
    test_pred_df.index.name = 'feature_num'
    print("\n测试集预测结果:")
    print(test_pred_df)

    for key in test_pred_df.columns:
        plt.plot(test_pred_df[key].values, label = key)
    plt.legend()
    plt.show()

    for key in test_pred_df.columns:
        plt.plot(train_acc_df[key].values, label = key)
    plt.legend()
    plt.show()


## 量化各个方法的最佳留一验证结果

### svm

In [44]:
method_res = pd.read_csv('/home/rsun@ZHANGroup.local/hqy_new/phd_thesis/result_method_compare/test_pred_SVM.csv')

# ---- 找最优特征数（留一验证准确率最高的列）----
feat_cols = [str(i) for i in range(1, 31)]

method_res = method_res.set_index('species') if 'species' in method_res.columns else method_res

pred_df = method_res[feat_cols].astype(int)
label_s = method_res['label'].astype(int)

# 每个 feature_num 下的准确率
acc_per_feat = (pred_df == label_s.values.reshape(-1, 1)).mean(axis=0)
acc_per_feat.index = acc_per_feat.index.astype(int)
acc_per_feat = acc_per_feat.sort_index()

best_acc       = acc_per_feat.max()
# 找出全部达到最高准确率的特征数（可能多个）
best_feat_nums = acc_per_feat[acc_per_feat == best_acc].index.tolist()

print(f'最高留一验证准确率: {best_acc:.4f}')
print(f'对应特征数 (共 {len(best_feat_nums)} 个): {best_feat_nums}')
# print()
# print('各特征数下准确率:')
# print(acc_per_feat.to_string())

# ---- 对每个最优特征数分别汇报出错物种 ----
for feat_num in best_feat_nums:
    col        = str(feat_num)
    error_mask = pred_df[col] != label_s
    error_df   = method_res.loc[error_mask, [col, 'label']].copy()
    error_df.columns = ['pred', 'true_label']
    print(f'\n特征数={feat_num} 时预测出错的物种 (共 {error_mask.sum()} 个):')
    print(error_df.to_string())


最高留一验证准确率: 0.9423
对应特征数 (共 1 个): [25]

特征数=25 时预测出错的物种 (共 6 个):
                       pred  true_label
species                                
Echinops_telfairi         0           1
Physeter_catodon          0           1
Rousettus_aegyptiacus     0           1
Sorex_araneus             0           1
ZWS                       0           1
shrew_mole                0           1


### lg

In [18]:

method_res = pd.read_csv('/home/rsun@ZHANGroup.local/hqy_new/phd_thesis/result_method_compare/test_pred_LogisticRegression.csv')

# ---- 找最优特征数（留一验证准确率最高的列）----
feat_cols = [str(i) for i in range(1, 31)]

method_res = method_res.set_index('species') if 'species' in method_res.columns else method_res

pred_df = method_res[feat_cols].astype(int)
label_s = method_res['label'].astype(int)

# 每个 feature_num 下的准确率
acc_per_feat = (pred_df == label_s.values.reshape(-1, 1)).mean(axis=0)
acc_per_feat.index = acc_per_feat.index.astype(int)
acc_per_feat = acc_per_feat.sort_index()

best_acc       = acc_per_feat.max()
# 找出全部达到最高准确率的特征数（可能多个）
best_feat_nums = acc_per_feat[acc_per_feat == best_acc].index.tolist()

print(f'最高留一验证准确率: {best_acc:.4f}')
print(f'对应特征数 (共 {len(best_feat_nums)} 个): {best_feat_nums}')
# print()
# print('各特征数下准确率:')
# print(acc_per_feat.to_string())

# ---- 对每个最优特征数分别汇报出错物种 ----
for feat_num in best_feat_nums:
    col        = str(feat_num)
    error_mask = pred_df[col] != label_s
    error_df   = method_res.loc[error_mask, [col, 'label']].copy()
    error_df.columns = ['pred', 'true_label']
    print(f'\n特征数={feat_num} 时预测出错的物种 (共 {error_mask.sum()} 个):')
    print(error_df.to_string())


最高留一验证准确率: 0.9423
对应特征数 (共 2 个): [1, 30]

特征数=1 时预测出错的物种 (共 6 个):
                         pred  true_label
species                                  
Condylura_cristata          1           0
Echinops_telfairi           0           1
Ochotona_princeps           1           0
Panthera_tigris_altaica     1           0
Physeter_catodon            0           1
ZWS                         0           1

特征数=30 时预测出错的物种 (共 6 个):
                         pred  true_label
species                                  
Echinops_telfairi           0           1
Panthera_tigris_altaica     1           0
Rousettus_aegyptiacus       0           1
Sorex_araneus               0           1
ZWS                         0           1
shrew_mole                  0           1


### rf

In [24]:
method_res = pd.read_csv('/home/rsun@ZHANGroup.local/hqy_new/phd_thesis/result_method_compare/test_pred_RandomForest.csv')


#method_res = pd.read_csv('/home/rsun@ZHANGroup.local/hqy_new/phd_thesis/result_method_compare/test_pred_NaiveBayes(Categorical).csv')


#method_res = pd.read_csv('/home/rsun@ZHANGroup.local/hqy_new/phd_thesis/result_method_compare/test_pred_LogisticRegression.csv')

# ---- 找最优特征数（留一验证准确率最高的列）----
feat_cols = [str(i) for i in range(1, 31)]

method_res = method_res.set_index('species') if 'species' in method_res.columns else method_res

pred_df = method_res[feat_cols].astype(int)
label_s = method_res['label'].astype(int)

# 每个 feature_num 下的准确率
acc_per_feat = (pred_df == label_s.values.reshape(-1, 1)).mean(axis=0)
acc_per_feat.index = acc_per_feat.index.astype(int)
acc_per_feat = acc_per_feat.sort_index()

best_acc       = acc_per_feat.max()
# 找出全部达到最高准确率的特征数（可能多个）
best_feat_nums = acc_per_feat[acc_per_feat == best_acc].index.tolist()

print(f'最高留一验证准确率: {best_acc:.4f}')
print(f'对应特征数 (共 {len(best_feat_nums)} 个): {best_feat_nums}')
# ---- 对每个最优特征数分别汇报出错物种 ----
for feat_num in best_feat_nums:
    col        = str(feat_num)
    error_mask = pred_df[col] != label_s
    error_df   = method_res.loc[error_mask, [col, 'label']].copy()
    error_df.columns = ['pred', 'true_label']
    print(f'\n特征数={feat_num} 时预测出错的物种 (共 {error_mask.sum()} 个):')
    print(error_df.to_string())


最高留一验证准确率: 0.9519
对应特征数 (共 1 个): [2]

特征数=2 时预测出错的物种 (共 5 个):
                      pred  true_label
species                               
Condylura_cristata       1           0
Echinops_telfairi        0           1
Hipposideros_armiger     0           1
Physeter_catodon         0           1
ZWS                      0           1


### NB

In [26]:
method_res = pd.read_csv('/home/rsun@ZHANGroup.local/hqy_new/phd_thesis/result_method_compare/test_pred_NaiveBayes(Categorical).csv')


#method_res = pd.read_csv('/home/rsun@ZHANGroup.local/hqy_new/phd_thesis/result_method_compare/test_pred_LogisticRegression.csv')

# ---- 找最优特征数（留一验证准确率最高的列）----
feat_cols = [str(i) for i in range(1, 31)]

method_res = method_res.set_index('species') if 'species' in method_res.columns else method_res

pred_df = method_res[feat_cols].astype(int)
label_s = method_res['label'].astype(int)

# 每个 feature_num 下的准确率
acc_per_feat = (pred_df == label_s.values.reshape(-1, 1)).mean(axis=0)
acc_per_feat.index = acc_per_feat.index.astype(int)
acc_per_feat = acc_per_feat.sort_index()

best_acc       = acc_per_feat.max()
# 找出全部达到最高准确率的特征数（可能多个）
best_feat_nums = acc_per_feat[acc_per_feat == best_acc].index.tolist()

print(f'最高留一验证准确率: {best_acc:.4f}')
print(f'对应特征数 (共 {len(best_feat_nums)} 个): {best_feat_nums}')
# ---- 对每个最优特征数分别汇报出错物种 ----
for feat_num in best_feat_nums:
    col        = str(feat_num)
    error_mask = pred_df[col] != label_s
    error_df   = method_res.loc[error_mask, [col, 'label']].copy()
    error_df.columns = ['pred', 'true_label']
    print(f'\n特征数={feat_num} 时预测出错的物种 (共 {error_mask.sum()} 个):')
    print(error_df.to_string())


最高留一验证准确率: 0.9519
对应特征数 (共 2 个): [1, 5]

特征数=1 时预测出错的物种 (共 5 个):
                    pred  true_label
species                             
Condylura_cristata     1           0
Echinops_telfairi      0           1
Lipotes_vexillifer     0           1
Physeter_catodon       0           1
ZWS                    0           1

特征数=5 时预测出错的物种 (共 5 个):
                         pred  true_label
species                                  
Echinops_telfairi           0           1
Panthera_tigris_altaica     1           0
Physeter_catodon            0           1
Rousettus_aegyptiacus       0           1
Sorex_araneus               0           1
